# Extract CelebDFv1 Level 5 Features


In [1]:
!pip install -q open_clip_torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00


In [2]:
from pathlib import Path

import open_clip
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm


In [3]:
CSV_PATH = "/kaggle/input/datasets/jamestashvik/deepfakebench/deepfakebench_dataset.csv"
DEEPFAKEBENCH_ROOT = "/kaggle/input/datasets/jamestashvik/deepfakebench/DeepFakeBench"
CELEBDFV1_ORIGINAL_ROOT = f"{DEEPFAKEBENCH_ROOT}/Celeb-DF-v1"
CELEBDFV1_LEVEL1_ROOT = "/kaggle/input/datasets/elisevo/celebdfv1-level-1/processed_output"

TRANSFORM_ROOTS = {
    "color_contrast": f"{CELEBDFV1_LEVEL1_ROOT}/color_contrast/level_1/Celeb-DF-v1",
    "color_saturation": f"{CELEBDFV1_LEVEL1_ROOT}/color_saturation/level_1/Celeb-DF-v1",
    "gaussian_blur": f"{CELEBDFV1_LEVEL1_ROOT}/gaussian_blur/level_1/Celeb-DF-v1",
    "resize": f"{CELEBDFV1_LEVEL1_ROOT}/resize/level_1/Celeb-DF-v1",
}

OUTPUT_DIR = Path("/kaggle/working/celebdfv1_level1_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


'cuda'

In [4]:
df = pd.read_csv(CSV_PATH)
df = df[df["datasetname"] == "Celeb-DF-v1"].copy()
df["label_num"] = df["label"].map({"REAL": 0, "FAKE": 1}).astype(int)
df["imagepath_fixed"] = df["imagepath"].astype(str).str.replace(
    "../input/deepfakebench",
    DEEPFAKEBENCH_ROOT,
    regex=False,
)
df = df.reset_index(drop=True)

print(df.shape)
print(df["label_num"].value_counts())
df.head()


(38312, 7)
label_num
1    33308
0     5004
Name: count, dtype: int64


,imagepath,original_width,original_height,label,datasetname,label_num,imagepath_fixed
0,../input/deepfakebench/Celeb-DF-v1/YouTube-rea...,256,256,FAKE,Celeb-DF-v1,1,/kaggle/input/datasets/jamestashvik/deepfakebe...
1,../input/deepfakebench/Celeb-DF-v1/YouTube-rea...,256,256,FAKE,Celeb-DF-v1,1,/kaggle/input/datasets/jamestashvik/deepfakebe...
2,../input/deepfakebench/Celeb-DF-v1/YouTube-rea...,256,256,FAKE,Celeb-DF-v1,1,/kaggle/input/datasets/jamestashvik/deepfakebe...
3,../input/deepfakebench/Celeb-DF-v1/YouTube-rea...,256,256,FAKE,Celeb-DF-v1,1,/kaggle/input/datasets/jamestashvik/deepfakebe...
4,../input/deepfakebench/Celeb-DF-v1/YouTube-rea...,256,256,FAKE,Celeb-DF-v1,1,/kaggle/input/datasets/jamestashvik/deepfakebe...


In [5]:
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14",
    pretrained="openai",
)

clip_model = clip_model.to(DEVICE).eval()


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [6]:
class ImageDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["imagepath_fixed"]).convert("RGB")
        image = preprocess(image)
        label = int(row["label_num"])
        path = row["imagepath_fixed"]
        return image, label, path


def extract_features(dataframe):
    loader = DataLoader(
        ImageDataset(dataframe),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    features = []
    labels = []
    paths = []

    with torch.no_grad():
        for images, batch_labels, batch_paths in tqdm(loader):
            images = images.to(DEVICE, non_blocking=True)
            batch_features = clip_model.encode_image(images)
            batch_features = F.normalize(batch_features, dim=-1)

            features.append(batch_features.cpu())
            labels.append(batch_labels.cpu())
            paths.extend(batch_paths)

    return torch.cat(features), torch.cat(labels), paths


In [7]:
saved_paths = []

for transform_name, transform_root in TRANSFORM_ROOTS.items():
    transform_df = df.copy()
    transform_df["imagepath_fixed"] = transform_df["imagepath_fixed"].str.replace(
        CELEBDFV1_ORIGINAL_ROOT,
        transform_root,
        regex=False,
    )

    output_path = OUTPUT_DIR / f"celebdfv1_level1_{transform_name}_features.pt"

    print(f"\nExtracting {transform_name}")
    print("root:", transform_root)
    print("output:", output_path)

    features, labels, paths = extract_features(transform_df)

    torch.save(
        {
            "features": features,
            "labels": labels,
            "paths": paths,
            "dataset_name": "Celeb-DF-v1",
            "transform_name": transform_name,
            "transform_level": 1,
            "clip_model": "ViT-L-14/openai",
        },
        output_path,
    )

    print("features:", features.shape)
    print("labels:", labels.shape)
    print("saved to:", output_path)
    saved_paths.append(output_path)

saved_paths



Extracting color_contrast
root: /kaggle/input/datasets/elisevo/celebdfv1-level-1/processed_output/color_contrast/level_1/Celeb-DF-v1
output: /kaggle/working/celebdfv1_level1_features/celebdfv1_level1_color_contrast_features.pt


100%|██████████| 599/599 [31:05<00:00,  3.11s/it]


features: torch.Size([38312, 768])
labels: torch.Size([38312])
saved to: /kaggle/working/celebdfv1_level1_features/celebdfv1_level1_color_contrast_features.pt

Extracting color_saturation
root: /kaggle/input/datasets/elisevo/celebdfv1-level-1/processed_output/color_saturation/level_1/Celeb-DF-v1
output: /kaggle/working/celebdfv1_level1_features/celebdfv1_level1_color_saturation_features.pt


100%|██████████| 599/599 [31:19<00:00,  3.14s/it]


features: torch.Size([38312, 768])
labels: torch.Size([38312])
saved to: /kaggle/working/celebdfv1_level1_features/celebdfv1_level1_color_saturation_features.pt

Extracting gaussian_blur
root: /kaggle/input/datasets/elisevo/celebdfv1-level-1/processed_output/gaussian_blur/level_1/Celeb-DF-v1
output: /kaggle/working/celebdfv1_level1_features/celebdfv1_level1_gaussian_blur_features.pt


100%|██████████| 599/599 [31:20<00:00,  3.14s/it]


features: torch.Size([38312, 768])
labels: torch.Size([38312])
saved to: /kaggle/working/celebdfv1_level1_features/celebdfv1_level1_gaussian_blur_features.pt

Extracting resize
root: /kaggle/input/datasets/elisevo/celebdfv1-level-1/processed_output/resize/level_1/Celeb-DF-v1
output: /kaggle/working/celebdfv1_level1_features/celebdfv1_level1_resize_features.pt


100%|██████████| 599/599 [31:19<00:00,  3.14s/it]


features: torch.Size([38312, 768])
labels: torch.Size([38312])
saved to: /kaggle/working/celebdfv1_level1_features/celebdfv1_level1_resize_features.pt


[PosixPath('/kaggle/working/celebdfv1_level1_features/celebdfv1_level1_color_contrast_features.pt'),
 PosixPath('/kaggle/working/celebdfv1_level1_features/celebdfv1_level1_color_saturation_features.pt'),
 PosixPath('/kaggle/working/celebdfv1_level1_features/celebdfv1_level1_gaussian_blur_features.pt'),
 PosixPath('/kaggle/working/celebdfv1_level1_features/celebdfv1_level1_resize_features.pt')]